In [2]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)
print("Processed directory exists:", PROCESSED_DIR.exists())

Project root: d:\Internship tasks\hiver-support-agent
Processed directory: d:\Internship tasks\hiver-support-agent\data\processed
Processed directory exists: True


In [3]:
print("Processed files:\n")

for path in PROCESSED_DIR.iterdir():
    print(path.name)

Processed files:

retrieval_corpus.parquet


In [4]:
DATA_DIR = PROJECT_ROOT / "data"

print("Data directory:", DATA_DIR)
print("\nFiles and folders:")

for path in DATA_DIR.rglob("*"):
    print(path.relative_to(DATA_DIR))

Data directory: d:\Internship tasks\hiver-support-agent\data

Files and folders:
amazonhelp_conversations.csv
amazonhelp_intent_sample.csv
processed
raw
twcs.db
processed\retrieval_corpus.parquet


In [5]:
CONVERSATIONS_FILE = DATA_DIR / "amazonhelp_conversations.csv"

print("Loading:", CONVERSATIONS_FILE)

amazon_conversations = pd.read_csv(CONVERSATIONS_FILE)

print("Loaded successfully!")
print("Rows:", len(amazon_conversations))
print("Columns:", amazon_conversations.columns.tolist())

Loading: d:\Internship tasks\hiver-support-agent\data\amazonhelp_conversations.csv
Loaded successfully!
Rows: 374042
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'conversation_id']


In [6]:
# Keep only customer messages
customer_messages = amazon_conversations[
    amazon_conversations["inbound"] == True
].copy()

print("Customer messages:", len(customer_messages))
print("Columns:", customer_messages.columns.tolist())

Customer messages: 203598
Columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'conversation_id']


In [7]:
support_cases = customer_messages[
    customer_messages["response_tweet_id"].notna()
].copy()

print("Support cases:", len(support_cases))
print("Unique conversations:", support_cases["conversation_id"].nunique())
print("Unique customer tweets:", support_cases["tweet_id"].nunique())

Support cases: 168466
Unique conversations: 82534
Unique customer tweets: 168466


In [8]:
response_lookup = amazon_conversations[
    ["tweet_id", "author_id", "inbound", "created_at", "text"]
].copy()

response_lookup = response_lookup.rename(
    columns={
        "tweet_id": "response_tweet_id",
        "author_id": "response_author_id",
        "inbound": "response_inbound",
        "created_at": "response_created_at",
        "text": "response_text",
    }
)

print("Response lookup rows:", len(response_lookup))
print(response_lookup.head(3).to_string())

Response lookup rows: 374042
   response_tweet_id response_author_id  response_inbound        response_created_at                                                                                                                   response_text
0                272             115770              True  2017-11-22 09:14:39+00:00                                                                                                        amazonのfireTVstickが見れない😢
1                269         AmazonHelp             False  2017-11-22 09:23:01+00:00  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
2                270             115770              True  2017-11-22 09:24:30+00:00                                                                                   @AmazonHelp ありがとうございます。\n今、電話で主人が対応していただいてます。


In [9]:
# Make both merge keys the same type
support_cases["response_tweet_id"] = pd.to_numeric(
    support_cases["response_tweet_id"],
    errors="coerce"
).astype("Int64")

response_lookup["response_tweet_id"] = pd.to_numeric(
    response_lookup["response_tweet_id"],
    errors="coerce"
).astype("Int64")


# Now perform the merge
retrieval_corpus = support_cases.merge(
    response_lookup,
    on="response_tweet_id",
    how="left"
)

print("Retrieval corpus rows:", len(retrieval_corpus))
print("Columns:")
print(retrieval_corpus.columns.tolist())

Retrieval corpus rows: 168466
Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'conversation_id', 'response_author_id', 'response_inbound', 'response_created_at', 'response_text']


In [10]:
print(
    "Missing responses:",
    retrieval_corpus["response_text"].isna().sum()
)

print(
    "Total rows:",
    len(retrieval_corpus)
)

Missing responses: 19909
Total rows: 168466


In [11]:
missing_responses = retrieval_corpus[
    retrieval_corpus["response_text"].isna()
].copy()

print("Missing response rows:", len(missing_responses))

print("\nSample missing response IDs:")
print(
    missing_responses[
        ["tweet_id", "response_tweet_id", "conversation_id", "text"]
    ].head(10).to_string()
)

Missing response rows: 19909

Sample missing response IDs:
     tweet_id  response_tweet_id  conversation_id                                                                                                                                                                                                                text
31        672               <NA>            672.0                                                                           I'm NEVER using Amazon again! After waiting in all day as item is "out for delivery", they've only gone and sent it to the WRONG COUNTRY!
34        678               <NA>            678.0                                                                         My package from @115821 with my Halloween costume was “delivered” Friday but I don’t have I so searching everywhere for a last minute idea.
39        682               <NA>            682.0                                                                                                          

In [12]:
all_tweet_ids = set(
    pd.to_numeric(
        amazon_conversations["tweet_id"],
        errors="coerce"
    ).dropna().astype("int64")
)

missing_response_ids = set(
    missing_responses["response_tweet_id"]
    .dropna()
    .astype("int64")
)

found_anywhere = missing_response_ids & all_tweet_ids

print("Unique missing response IDs:", len(missing_response_ids))
print("Found in amazon_conversations:", len(found_anywhere))
print("Not found in amazon_conversations:", len(missing_response_ids - all_tweet_ids))

Unique missing response IDs: 0
Found in amazon_conversations: 0
Not found in amazon_conversations: 0


In [13]:
print(
    retrieval_corpus[
        ["text", "response_text"]
    ].head(5).to_string()
)

                                                                    text                                                                                                                                              response_text
0                                               amazonのfireTVstickが見れない😢                             @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
1       @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。                                                                                      @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
2                                          @AmazonHelp こちらこそありがとうございました。                                                                                                     @115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET
3                                               amazonプライムビデオ、再生エラーが多いです  @115792 ご不便をおか

In [14]:
print("Retrieval corpus:", len(retrieval_corpus))

print("\nRetrieval corpus columns:")
print(retrieval_corpus.columns.tolist())

print("\nMissing response_text:")
print(retrieval_corpus["response_text"].isna().sum())

print("\nSample:")
print(
    retrieval_corpus[
        ["text", "response_text"]
    ].head(5).to_string()
)

Retrieval corpus: 168466

Retrieval corpus columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'conversation_id', 'response_author_id', 'response_inbound', 'response_created_at', 'response_text']

Missing response_text:
19909

Sample:
                                                                    text                                                                                                                                              response_text
0                                               amazonのfireTVstickが見れない😢                             @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
1       @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。                                                                                      @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET
2                

In [15]:
retrieval_corpus = retrieval_corpus.dropna(
    subset=["response_text"]
).copy()

retrieval_corpus = retrieval_corpus[
    retrieval_corpus["text"].notna()
].copy()

print("Final retrieval corpus:", len(retrieval_corpus))
print("Missing customer text:", retrieval_corpus["text"].isna().sum())
print("Missing response text:", retrieval_corpus["response_text"].isna().sum())

Final retrieval corpus: 148557
Missing customer text: 0
Missing response text: 0


In [16]:
retrieval_docs = retrieval_corpus[
    [
        "tweet_id",
        "conversation_id",
        "text",
        "response_text"
    ]
].copy()

print("Retrieval documents:", len(retrieval_docs))
print("Columns:", retrieval_docs.columns.tolist())

Retrieval documents: 148557
Columns: ['tweet_id', 'conversation_id', 'text', 'response_text']


In [17]:
retrieval_docs = retrieval_docs.rename(
    columns={
        "tweet_id": "customer_tweet_id",
        "text": "customer_query",
        "response_text": "agent_response"
    }
)

print(retrieval_docs.columns.tolist())

['customer_tweet_id', 'conversation_id', 'customer_query', 'agent_response']


In [18]:
duplicate_count = retrieval_docs.duplicated(
    subset=["customer_query", "agent_response"]
).sum()

print("Duplicate customer-response pairs:", duplicate_count)

Duplicate customer-response pairs: 1


In [19]:
retrieval_docs = retrieval_docs.drop_duplicates(
    subset=["customer_query", "agent_response"]
).reset_index(drop=True)

print("Retrieval documents after deduplication:", len(retrieval_docs))

Retrieval documents after deduplication: 148556


In [20]:
retrieval_docs["document"] = (
    "Customer issue: "
    + retrieval_docs["customer_query"].astype(str)
    + "\n\n"
    + "AmazonHelp response: "
    + retrieval_docs["agent_response"].astype(str)
)

print(retrieval_docs[["customer_query", "agent_response", "document"]].head(3).to_string())

                                                     customer_query                                                                                                                  agent_response                                                                                                                                                                                         document
0                                          amazonのfireTVstickが見れない😢  @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET  Customer issue: amazonのfireTVstickが見れない😢\n\nAmazonHelp response: @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
1  @AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるんでしょうね。                                                           @115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただきありがとうございました。ET                   Customer

In [21]:
print("Rows:", len(retrieval_docs))
print("Columns:", retrieval_docs.columns.tolist())

print("\nEmpty customer queries:",
      (retrieval_docs["customer_query"].str.strip() == "").sum())

print("Empty agent responses:",
      (retrieval_docs["agent_response"].str.strip() == "").sum())

print("\nAverage document length:",
      retrieval_docs["document"].str.len().mean())

print("Maximum document length:",
      retrieval_docs["document"].str.len().max())

Rows: 148556
Columns: ['customer_tweet_id', 'conversation_id', 'customer_query', 'agent_response', 'document']

Empty customer queries: 0
Empty agent responses: 0

Average document length: 278.10521284902666
Maximum document length: 698


In [22]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "retrieval_corpus.parquet"

retrieval_docs.to_parquet(
    output_path,
    index=False
)

print("Saved successfully!")
print("Path:", output_path)
print("Rows:", len(retrieval_docs))

Saved successfully!
Path: d:\Internship tasks\hiver-support-agent\data\processed\retrieval_corpus.parquet
Rows: 148556


In [23]:
print("File exists:", output_path.exists())
print("File size (MB):", round(output_path.stat().st_size / (1024 * 1024), 2))

File exists: True
File size (MB): 45.63


In [24]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: BAAI/bge-small-en-v1.5
Embedding dimension: 384


C:\Users\rites\AppData\Local\Temp\ipykernel_1096\1890869545.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [25]:
test_documents = retrieval_docs["document"].head(100).tolist()

test_embeddings = model.encode(
    test_documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", test_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (100, 384)


In [26]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    test_embeddings[:1],
    test_embeddings
)[0]

top_indices = similarities.argsort()[-5:][::-1]

for idx in top_indices:
    print("=" * 70)
    print("Similarity:", round(similarities[idx], 4))
    print("Customer:", retrieval_docs.iloc[idx]["customer_query"])
    print("Response:", retrieval_docs.iloc[idx]["agent_response"][:300])

Similarity: 1.0
Customer: amazonのfireTVstickが見れない😢
Response: @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET
Similarity: 0.9616
Customer: @116313 fire TV stickを購入したのですが一瞬しかうつりません.°(ಗдಗ。)°.どうしたらいいのでしょうか...
Response: @116312 アマゾンです。お困りの状況が特定できないのですが、エラーメッセージや画面の状況、操作方法など詳しい状況を教えていただけますか。ご利用端末はテレビですかPS4などのゲーム機でしょうか。ET
Similarity: 0.9537
Customer: @AmazonHelp 先日教えていただいたように、fireTVstickとルーターの両方を再起動してみたら、前回のようには途切れなくなりました！ありがとうございます😊今の所大丈夫ですが、あとは72時間TV当日に接続が切れてしまわないように祈るのみです...。
お礼と報告まで。 https://t.co/hS5VmxuIfX
Response: @116312 わざわざご連絡をいただき、ありがとうございます。問題が改善されたようで、何よりでございます。万が一再度お困りのことがございましたら、ご遠慮なくお知らせください🙏 YM
Similarity: 0.9512
Customer: @116313 FireTVstick購入し接続完了、画面の切り替えも問題ないのですが、音声が切り替わらないです。どうしたらいいのでしょうか？(＞＜)
Response: @118502 Fire TV Stickのご注文ありがとうございます。
音声が切り替わらないとのことなので、設定等を確認の上ご案内させていただきます。
よろしければ、下記URLよりご連絡ください。https://t.co/bwBU0NvYIn TY
Similarity: 0.8863
Customer: @AmazonHelp ありがとうございます！
テレビにstickを接続してい

In [27]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

retrieval_path = PROCESSED_DIR / "retrieval_corpus.parquet"

retrieval_docs = pd.read_parquet(retrieval_path)

print("Retrieval corpus loaded!")
print("Rows:", len(retrieval_docs))
print("Columns:", retrieval_docs.columns.tolist())

Retrieval corpus loaded!
Rows: 148556
Columns: ['customer_tweet_id', 'conversation_id', 'customer_query', 'agent_response', 'document']


In [28]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-small-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("Embedding model loaded!")
print("Model:", MODEL_NAME)
print("Embedding dimension:", model.get_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded!
Model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


In [29]:
documents = retrieval_docs["document"].tolist()

print("Documents:", len(documents))
print("First document:")
print(documents[0][:300])

Documents: 148556
First document:
Customer issue: amazonのfireTVstickが見れない😢

AmazonHelp response: @115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況でしょうか。一般的なトラブルシューティングを記載したヘルプがございますので、ご参照ください。https://t.co/2pbG55qJ7h ET


In [30]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU available to PyTorch")

PyTorch version: 2.14.0+cpu
CUDA available: False
No CUDA GPU available to PyTorch


In [31]:
import time

test_documents = documents[:1000]

start = time.time()

test_embeddings = model.encode(
    test_documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

elapsed = time.time() - start

print("\nBenchmark complete!")
print("Documents:", len(test_documents))
print("Embedding shape:", test_embeddings.shape)
print("Time:", round(elapsed, 2), "seconds")
print("Estimated time for full corpus:",
      round(elapsed * len(documents) / 1000 / 60, 2), "minutes")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Benchmark complete!
Documents: 1000
Embedding shape: (1000, 384)
Time: 49.7 seconds
Estimated time for full corpus: 123.05 minutes


In [32]:
print("Total documents:", len(retrieval_docs))

unique_documents = retrieval_docs["document"].nunique()

print("Unique documents:", unique_documents)
print(
    "Duplicate documents:",
    len(retrieval_docs) - unique_documents
)

Total documents: 148556
Unique documents: 148556
Duplicate documents: 0


In [33]:
from fastembed import TextEmbedding

print("Loading FastEmbed model...")

fast_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

print("FastEmbed model loaded!")

Loading FastEmbed model...
FastEmbed model loaded!


In [34]:
import time
import numpy as np

test_documents = documents[:1000]

start = time.time()

test_embeddings_fast = np.array(
    list(fast_model.embed(test_documents))
)

elapsed = time.time() - start

print("\nFastEmbed benchmark complete!")
print("Documents:", len(test_documents))
print("Embedding shape:", test_embeddings_fast.shape)
print("Time:", round(elapsed, 2), "seconds")
print(
    "Estimated time for full corpus:",
    round(elapsed * len(documents) / 1000 / 60, 2),
    "minutes"
)


FastEmbed benchmark complete!
Documents: 1000
Embedding shape: (1000, 384)
Time: 76.74 seconds
Estimated time for full corpus: 189.99 minutes


In [35]:
import pandas as pd

golden_path = PROJECT_ROOT / "notebooks" / "amazonhelp_golden_set_raw.csv"

golden_df = pd.read_csv(golden_path)

print("Golden set rows:", len(golden_df))

# Detect Japanese characters
japanese_mask = golden_df["text"].astype(str).str.contains(
    "[ぁ-んァ-ン一-龥]",
    regex=True,
    na=False
)

print("Japanese messages:", japanese_mask.sum())
print("Non-Japanese messages:", (~japanese_mask).sum())
print(
    "Japanese percentage:",
    round(japanese_mask.mean() * 100, 2),
    "%"
)

Golden set rows: 200
Japanese messages: 17
Non-Japanese messages: 183
Japanese percentage: 8.5 %


In [36]:
print("\nExamples of Japanese messages:\n")

print(
    golden_df.loc[japanese_mask, "text"]
    .head(10)
    .to_string(index=False)
)


Examples of Japanese messages:

                 画集あまぞんくんで頼んでるんだけどええい！支払い番号はまだか…！！
ぎゃあああ、Amazonまでクリック直前の画面縦ズレしやがった！　プライム体験とかいらんかった...
               は？P3・P4・P4G・P5のサントラ全部無料かよ amazon神だな
                          あー、Amazon……オフラインでも読めるんかな
届いた〜（｡ӧ◡ӧ｡）\nAmazonプライム会員だから\nものっそい安く買えた♡♡\nもうほ...
                                 Amazonの使い方が分からない。
    もアマゾンで荷物頼んだけど今日配達予定だけど郵便の追跡見てみると12月6日になっとるんよね〜
     アマゾンでお菓子買ったのにパソコンが来た… https://t.co/Q4Fh6bKCR2
Amazon（ @116313 ）アプリのiPad版が縦画面固定になっちゃったのは儂だけではな...
                                  やはりamazon詐欺であったか


In [37]:
print("Retrieval documents:", len(retrieval_docs))
print("Golden set:", len(golden_df))
print("Current embedding model:", model)

Retrieval documents: 148556
Golden set: 200
Current embedding model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


In [38]:
golden_queries = golden_df["text"].astype(str).tolist()

print("Golden queries:", len(golden_queries))
print("First query:", golden_queries[0])


Golden queries: 200
First query: @115850 @115821 It's been 11 days since I order a product from you people, I didn't even get a single response from the delivery team. What I can see in my page is delivery unsuccessful !!! It's not acceptable at all !!!


In [39]:
import time

start = time.time()

english_golden_embeddings = model.encode(
    golden_queries,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start

print("English model benchmark complete!")
print("Shape:", english_golden_embeddings.shape)
print("Time:", round(elapsed, 2), "seconds")

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

English model benchmark complete!
Shape: (200, 384)
Time: 4.07 seconds


In [40]:
print("Current model:", model)
print("Golden set:", len(golden_df))
print("Japanese queries:", japanese_mask.sum())

Current model: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
Golden set: 200
Japanese queries: 17


In [41]:
import time

# Use a small sample for benchmarking
benchmark_documents = retrieval_docs["document"].head(2000).tolist()

batch_sizes = [32, 64, 128, 256]

print("Testing batch sizes...\n")

for batch_size in batch_sizes:
    start = time.time()

    test_embeddings = model.encode(
        benchmark_documents,
        batch_size=batch_size,
        show_progress_bar=False,
        normalize_embeddings=True
    )

    elapsed = time.time() - start

    print(
        f"Batch size {batch_size}: "
        f"{elapsed:.2f} seconds"
    )

Testing batch sizes...

Batch size 32: 78.68 seconds
Batch size 64: 99.74 seconds
Batch size 128: 116.84 seconds
Batch size 256: 124.88 seconds


In [42]:
import time
import numpy as np

print("Starting full-corpus embedding generation...")
print("Documents:", len(retrieval_docs))
print("Batch size: 32")

start = time.time()

embeddings = model.encode(
    retrieval_docs["document"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start

print("\nEmbedding generation complete!")
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("Time:", round(elapsed / 60, 2), "minutes")

Starting full-corpus embedding generation...
Documents: 148556
Batch size: 32


Batches:   0%|          | 0/4643 [00:00<?, ?it/s]


Embedding generation complete!
Shape: (148556, 384)
Data type: float32
Time: 129.73 minutes


In [43]:
import numpy as np
from pathlib import Path

EMBEDDINGS_PATH = PROCESSED_DIR / "retrieval_embeddings.npy"

np.save(EMBEDDINGS_PATH, embeddings)

print("Embeddings saved successfully!")
print("Path:", EMBEDDINGS_PATH)
print("Shape:", embeddings.shape)
print("File size (MB):", round(EMBEDDINGS_PATH.stat().st_size / (1024 * 1024), 2))

Embeddings saved successfully!
Path: d:\Internship tasks\hiver-support-agent\data\processed\retrieval_embeddings.npy
Shape: (148556, 384)
File size (MB): 217.61


In [44]:
import numpy as np

loaded_embeddings = np.load(EMBEDDINGS_PATH)

print("Embeddings loaded successfully!")
print("Shape:", loaded_embeddings.shape)
print("Data type:", loaded_embeddings.dtype)
print("Same shape:", loaded_embeddings.shape == embeddings.shape)
print("Same values:", np.array_equal(loaded_embeddings, embeddings))

Embeddings loaded successfully!
Shape: (148556, 384)
Data type: float32
Same shape: True
Same values: True


In [46]:
import numpy as np
import faiss

# Use the correct project path
embeddings_path = PROCESSED_DIR / "retrieval_embeddings.npy"
index_path = PROCESSED_DIR / "retrieval.index"

# Load embeddings
embeddings = np.load(embeddings_path)

print("Embeddings loaded:", embeddings.shape)

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Add embeddings
index.add(embeddings)

print("FAISS index built successfully!")
print("Index dimension:", index.d)
print("Number of vectors:", index.ntotal)

# Save index
faiss.write_index(index, str(index_path))

print("FAISS index saved successfully!")
print("Path:", index_path)

Embeddings loaded: (148556, 384)
FAISS index built successfully!
Index dimension: 384
Number of vectors: 148556
FAISS index saved successfully!
Path: d:\Internship tasks\hiver-support-agent\data\processed\retrieval.index


In [47]:
import faiss

# Load the saved FAISS index
loaded_index = faiss.read_index(str(index_path))

print("FAISS index loaded successfully!")
print("Index dimension:", loaded_index.d)
print("Number of vectors:", loaded_index.ntotal)

print("Expected vectors:", len(retrieval_docs))
print("Vector count matches:",
      loaded_index.ntotal == len(retrieval_docs))

FAISS index loaded successfully!
Index dimension: 384
Number of vectors: 148556
Expected vectors: 148556
Vector count matches: True


In [48]:
query = "My Fire TV Stick is not working"

print("Query:", query)

Query: My Fire TV Stick is not working


In [49]:
# Convert query into an embedding
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

# Normalize query embedding
faiss.normalize_L2(query_embedding)

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [50]:
k = 5

scores, indices = loaded_index.search(query_embedding, k)

print("Top results:\n")

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    row = retrieval_docs.iloc[idx]

    print("=" * 70)
    print(f"Rank: {rank}")
    print(f"Similarity: {score:.4f}")
    print(f"Customer: {row['customer_query']}")
    print(f"Response: {row['agent_response'][:500]}")

Top results:

Rank: 1
Similarity: 0.8494
Customer: @128308 Does not work and you don't have customer support as well https://t.co/F3QCGd0pxG
Response: @357560 Sorry for the trouble, Aastha. We do have support for Fire TV Stick, please reach them here: https://t.co/KGw598bWlt ^JC
Rank: 2
Similarity: 0.8468
Customer: @116439 hello. My Fire stick has stopped working. Stuck on the logo screen. Can you help?
Response: @553967 I'm sorry your Fire Stick isn't working. Please unplug the power cable for 1 minute and then plug it back in. You may also use the remote to restart as well. For more information, you may visit this link: https://t.co/xDld9aaUw0 ^SJ
Rank: 3
Similarity: 0.8433
Customer: @AmazonHelp It's been the last week or so.
Response: @382602 Sorry for the Fire TV Stick troubles, Rich. Restarting your device can resolve many issues. Have you attempted restarting the device? https://t.co/4Eiq16nTZl ^TH
Rank: 4
Similarity: 0.8379
Customer: Why is my fire stick not workingggg
Response:

In [51]:
def retrieve_documents(query, k=5):
    """
    Retrieve the most similar support documents for a customer query.
    """

    # Create query embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Search FAISS
    scores, indices = loaded_index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        row = retrieval_docs.iloc[idx]

        results.append({
            "score": float(score),
            "customer_query": row["customer_query"],
            "agent_response": row["agent_response"],
            "conversation_id": row["conversation_id"]
        })

    return results

In [52]:
query = "My Fire TV Stick is not working"

results = retrieve_documents(query, k=5)

for i, result in enumerate(results, start=1):
    print("=" * 70)
    print(f"Rank: {i}")
    print(f"Similarity: {result['score']:.4f}")
    print(f"Customer: {result['customer_query']}")
    print(f"Response: {result['agent_response'][:300]}")

Rank: 1
Similarity: 0.8494
Customer: @128308 Does not work and you don't have customer support as well https://t.co/F3QCGd0pxG
Response: @357560 Sorry for the trouble, Aastha. We do have support for Fire TV Stick, please reach them here: https://t.co/KGw598bWlt ^JC
Rank: 2
Similarity: 0.8468
Customer: @116439 hello. My Fire stick has stopped working. Stuck on the logo screen. Can you help?
Response: @553967 I'm sorry your Fire Stick isn't working. Please unplug the power cable for 1 minute and then plug it back in. You may also use the remote to restart as well. For more information, you may visit this link: https://t.co/xDld9aaUw0 ^SJ
Rank: 3
Similarity: 0.8433
Customer: @AmazonHelp It's been the last week or so.
Response: @382602 Sorry for the Fire TV Stick troubles, Rich. Restarting your device can resolve many issues. Have you attempted restarting the device? https://t.co/4Eiq16nTZl ^TH
Rank: 4
Similarity: 0.8379
Customer: Why is my fire stick not workingggg
Response: @683856 I'm s

In [53]:
# Inspect golden set columns
print("Golden set columns:")
print(golden_df.columns.tolist())

print("\nGolden set rows:", len(golden_df))

Golden set columns:
['conversation_id', 'tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'message_length', 'conversation_length', 'message_length_group', 'conversation_length_group']

Golden set rows: 200


In [54]:
# Create evaluation queries from the golden set

evaluation_queries = golden_df[
    ["tweet_id", "response_tweet_id", "text", "conversation_id"]
].copy()

evaluation_queries = evaluation_queries.rename(
    columns={
        "tweet_id": "query_tweet_id",
        "response_tweet_id": "expected_response_tweet_id",
        "text": "query"
    }
)

print("Evaluation queries:", len(evaluation_queries))
print("Columns:", evaluation_queries.columns.tolist())

print("\nFirst evaluation query:")
print(evaluation_queries.iloc[0])

Evaluation queries: 200
Columns: ['query_tweet_id', 'expected_response_tweet_id', 'query', 'conversation_id']

First evaluation query:
query_tweet_id                                                          2538181
expected_response_tweet_id                                              2538178
query                         @115850 @115821 It's been 11 days since I orde...
conversation_id                                                       2538181.0
Name: 0, dtype: object


In [55]:
# Check how many golden queries exist in the retrieval corpus

corpus_tweet_ids = set(
    retrieval_docs["customer_tweet_id"].astype(str)
)

evaluation_queries["query_tweet_id"] = (
    evaluation_queries["query_tweet_id"].astype(str)
)

found_count = evaluation_queries["query_tweet_id"].isin(
    corpus_tweet_ids
).sum()

print("Golden queries:", len(evaluation_queries))
print("Found in retrieval corpus:", found_count)
print("Missing from retrieval corpus:",
      len(evaluation_queries) - found_count)

Golden queries: 200
Found in retrieval corpus: 165
Missing from retrieval corpus: 35


In [56]:
missing_queries = evaluation_queries[
    ~evaluation_queries["query_tweet_id"].isin(corpus_tweet_ids)
].copy()

print("Missing queries:", len(missing_queries))

print("\nMissing query examples:")
print(
    missing_queries[
        ["query_tweet_id", "expected_response_tweet_id", "query", "conversation_id"]
    ].head(10).to_string(index=False)
)

Missing queries: 35

Missing query examples:
query_tweet_id                                                          expected_response_tweet_id                                                                                                                                                                                                                                                                                                        query  conversation_id
         74438                                                                   74439,74440,74437                                                                                                                                                      @115817 this is the package I received today.  This is just the beginning of the “delivery” season!  🤨 Gotta do better! @115821 https://t.co/u1UTtcoi1C          74438.0
        485348                                                                       485349,485347                       

In [57]:
corpus_response_ids = set(
    retrieval_docs["customer_tweet_id"].astype(str)
)

missing_queries["expected_response_tweet_id"] = (
    missing_queries["expected_response_tweet_id"].astype(str)
)

response_found_count = missing_queries[
    "expected_response_tweet_id"
].isin(corpus_response_ids).sum()

print("Missing customer queries:", len(missing_queries))
print(
    "Expected response IDs found in corpus:",
    response_found_count
)
print(
    "Expected response IDs also missing:",
    len(missing_queries) - response_found_count
)

Missing customer queries: 35
Expected response IDs found in corpus: 0
Expected response IDs also missing: 35


In [58]:
# Keep only golden-set queries that exist in our retrieval corpus

evaluation_valid = evaluation_queries[
    evaluation_queries["query_tweet_id"].isin(corpus_tweet_ids)
].copy()

print("Total golden queries:", len(evaluation_queries))
print("Valid evaluation queries:", len(evaluation_valid))
print("Excluded queries:", len(evaluation_queries) - len(evaluation_valid))

Total golden queries: 200
Valid evaluation queries: 165
Excluded queries: 35


In [59]:
# Add the expected conversation ID for evaluation

evaluation_valid["conversation_id"] = (
    evaluation_valid["conversation_id"].astype(str)
)

retrieval_docs["conversation_id"] = (
    retrieval_docs["conversation_id"].astype(str)
)

print("Evaluation conversations:",
      evaluation_valid["conversation_id"].nunique())

print("Retrieval conversations:",
      retrieval_docs["conversation_id"].nunique())

Evaluation conversations: 165
Retrieval conversations: 77162


In [60]:
import numpy as np

def evaluate_retrieval(evaluation_df, k=10):
    hits = 0
    reciprocal_ranks = []

    for _, row in evaluation_df.iterrows():

        query = row["query"]
        expected_conversation = row["conversation_id"]

        # Embed query
        query_embedding = model.encode(
            [query],
            convert_to_numpy=True
        ).astype("float32")

        faiss.normalize_L2(query_embedding)

        # Search
        scores, indices = loaded_index.search(
            query_embedding,
            k
        )

        retrieved_conversations = [
            str(retrieval_docs.iloc[idx]["conversation_id"])
            for idx in indices[0]
        ]

        # Recall@K
        if expected_conversation in retrieved_conversations:
            hits += 1

        # Reciprocal rank
        rank = None

        for position, conversation_id in enumerate(
            retrieved_conversations,
            start=1
        ):
            if conversation_id == expected_conversation:
                rank = position
                break

        if rank is not None:
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    recall = hits / len(evaluation_df)
    mrr = np.mean(reciprocal_ranks)

    return recall, mrr

In [61]:
recall_5, mrr_5 = evaluate_retrieval(
    evaluation_valid,
    k=5
)

recall_10, mrr_10 = evaluate_retrieval(
    evaluation_valid,
    k=10
)

print("Retrieval Evaluation")
print("=" * 50)

print(f"Evaluated queries: {len(evaluation_valid)}")
print(f"Recall@5:  {recall_5:.4f}")
print(f"Recall@10: {recall_10:.4f}")
print(f"MRR@5:     {mrr_5:.4f}")
print(f"MRR@10:    {mrr_10:.4f}")

Retrieval Evaluation
Evaluated queries: 165
Recall@5:  0.8970
Recall@10: 0.9333
MRR@5:     0.8677
MRR@10:    0.8727


In [62]:
import json
from pathlib import Path

evaluation_dir = PROJECT_ROOT / "evaluation"
evaluation_dir.mkdir(exist_ok=True)

metrics = {
    "evaluated_queries": len(evaluation_valid),
    "total_golden_queries": len(evaluation_queries),
    "excluded_queries": len(evaluation_queries) - len(evaluation_valid),
    "recall_at_5": round(recall_5, 4),
    "recall_at_10": round(recall_10, 4),
    "mrr_at_5": round(mrr_5, 4),
    "mrr_at_10": round(mrr_10, 4)
}

metrics_path = evaluation_dir / "retrieval_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print("Evaluation metrics saved!")
print("Path:", metrics_path)

Evaluation metrics saved!
Path: d:\Internship tasks\hiver-support-agent\evaluation\retrieval_metrics.json


In [63]:
test_queries = [
    "My Fire TV Stick is not working",
    "I cannot track my package",
    "My order has not arrived",
    "I was charged twice for my order",
    "I want to return a product"
]

for query in test_queries:

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = loaded_index.search(
        query_embedding,
        3
    )

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        row = retrieval_docs.iloc[idx]

        print(f"\nRank {rank}")
        print("Similarity:", round(float(score), 4))
        print("Customer:", row["customer_query"][:250])
        print("Response:", row["agent_response"][:300])


QUERY: My Fire TV Stick is not working

Rank 1
Similarity: 0.8494
Customer: @128308 Does not work and you don't have customer support as well https://t.co/F3QCGd0pxG
Response: @357560 Sorry for the trouble, Aastha. We do have support for Fire TV Stick, please reach them here: https://t.co/KGw598bWlt ^JC

Rank 2
Similarity: 0.8468
Customer: @116439 hello. My Fire stick has stopped working. Stuck on the logo screen. Can you help?
Response: @553967 I'm sorry your Fire Stick isn't working. Please unplug the power cable for 1 minute and then plug it back in. You may also use the remote to restart as well. For more information, you may visit this link: https://t.co/xDld9aaUw0 ^SJ

Rank 3
Similarity: 0.8433
Customer: @AmazonHelp It's been the last week or so.
Response: @382602 Sorry for the Fire TV Stick troubles, Rich. Restarting your device can resolve many issues. Have you attempted restarting the device? https://t.co/4Eiq16nTZl ^TH

QUERY: I cannot track my package

Rank 1
Similarity: 0.

In [64]:
results = []

for query in test_queries:

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = loaded_index.search(
        query_embedding,
        3
    )

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        row = retrieval_docs.iloc[idx]

        results.append({
            "query": query,
            "rank": rank,
            "similarity": float(score),
            "customer_query": row["customer_query"],
            "agent_response": row["agent_response"]
        })

retrieval_results_df = pd.DataFrame(results)

results_path = evaluation_dir / "retrieval_examples.csv"

retrieval_results_df.to_csv(
    results_path,
    index=False
)

print("Retrieval examples saved!")
print("Rows:", len(retrieval_results_df))
print("Path:", results_path)

Retrieval examples saved!
Rows: 15
Path: d:\Internship tasks\hiver-support-agent\evaluation\retrieval_examples.csv
